# Fast Transformer Decoding: One Write-Head is All You Need

## 摘要
>原因在于反复加载庞大的"键"和"值"张量所带来的内存带宽开销。我们提出一种称为多查询注意力（multi-query attention）的变体，其中键和值在所有不同的注意力"头"之间共享，从而大幅缩减这些张量的大小，进而降低增量解码的内存带宽需求。实验验证表明，由此得到的模型解码速度确实大幅提升，且相比基线模型仅有轻微的质量下降。

## 背景

### 点积注意力机制：查询q和m对(k,v)键值对，输出y
$$
y = softmax(Q*K^{T})*V
$$

仅为一个单维的点积注意力
<p  align="center">
    <img src="./static/dotProduct.png">
</p>
其中，einsum是万能张量运算工具，einsum("输入1维度,输入2维度->输出维度", 张量1, 张量2)

In [12]:
import torch

def dot_attention(q,k,v,mask=None):
    d_k = q.size(-1)
    scores = torch.matmul(q, k.transpose(-2, -1)) / torch.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
    attn = torch.softmax(scores, dim=-1)
    output = torch.matmul(attn, v)
    return output, attn

### 多头注意力
> 并行训练 h 组独立的 Q/K/V，模型能同时从 h 个不同角度提取信息，点积注意力处理维度：
head_dim = embedding_dim / num_head
<p align="center">
    <img src="./static/multiHead.png">
</p>

In [ ]:
import torch.nn as nn

class MultiHeadAttention(nn.Module):
    def __init__(self, embedding_dim, num_heads):
        super().__init__()
        self.num_heads = num_heads
        self.embedding_dim = embedding_dim
        self.d_k = embedding_dim // num_heads
        
        self.W_q = nn.Linear(embedding_dim, embedding_dim)
        self.W_k = nn.Linear(embedding_dim, embedding_dim)
        self.W_v = nn.Linear(embedding_dim, embedding_dim)
        self.W_o = nn.Linear(embedding_dim, embedding_dim)

    def forward(self, q, k, v, mask=None):
        batch_size = q.size(0)
        
        # 线性投影生成 Q/K/V
        q = self.W_q(q)
        k = self.W_k(k)
        v = self.W_v(v)

        # 切分维度 (batch_size, num_heads, seq_len, d_k) 
        q = q.view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        k = k.view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        v = v.view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)

        attn_output, _ = dot_attention(q, k, v, mask)

        # (batch_size, seq_len, num_heads*d_k) → (batch_size, seq_len, embedding_dim)
        attn_output = attn_output.transpose(1, 2).contiguous().view(batch_size, -1, self.embedding_dim)

        output = self.W_o(attn_output)
        return output

### 批量多头性能分析
b：batch size（批量大小，一次喂多少条数据）

n：序列长度（一句话有多少词，自注意力中m=n）

d：模型总维度（embedding_dim，比如 512/1024）

h：注意力头数

k=v=d/h：单个头的维度（论文标准设置）

假设：源序列长度 = 目标序列长度(m = n)

**总算术运算量**（加减乘除总次数）为 $\Theta (bnd^2)$,一次矩阵乘法的运算量是abc,如 A(a,b)@B(b,c) 

**总内存访问量** $O(bnd + bhn^2 + d^2)$。一次矩阵读/写次数 = abc,如A.shape = (a,b,c)

bnd：输入词向量、Q/K/V、输出特征（主要数据），Q[b,h,n,k], hk=d ⇒ bhnk = bnd

$bhn^2$：注意力权重矩阵（多头的分数表），[b,h,n,n]

$d^2$：4 个投影矩阵 $W_q/W_k/W_v/W_o$（模型参数，很小）$W_q$ [h,d,k]

核心比值：内存访问 ÷ 算术运算 = $O(1/k + 1/bn)$

比值越小 → 算得多、读得少 → GPU 利用率拉满；

比值越大 → 算得少、读得多 → GPU 一直在等数据，性能极差。

### 增量多头注意力
> 后面的词依赖前面的所有词 → 这就是数据依赖，不能并行算所有词。


训练	并行批量计算	提前知道完整句子，所有词同时算注意力

生成	增量逐个计算	只能一个词一个词生成，不知道未来的词

如果生成时每次都重新计算所有词的 K/V，速度会极慢；

代码的解决方案：缓存之前所有词的 K 和 V，只计算当前新词的 K/V，拼接到缓存里 → 不用重复计算

In [ ]:
import torch
import torch.nn.functional as F

def MultiheadSelfAttentionIncremental(x, prev_K, prev_V, W_q, W_k, W_v, W_o, num_heads):
    """
    params：
        x: [batch_size, d_model] 
        prev_K: [batch_size, num_heads, m, d_k] 
        prev_V: [batch_size, num_heads, m, d_k] 
        W_q/W_k/W_v/W_o: 
        num_heads: 
    output:
        y: [batch_size, d_model]
        new_K: [batch_size, num_heads, m+1, d_k]
        new_V: [batch_size, num_heads, m+1, d_k]
    """
    batch_size, d_model = x.shape
    d_k = d_model // num_heads

    # 线性投影
    q = W_q(x)  # [b, d_model]
    k = W_k(x)  # [b, d_model]
    v = W_v(x)  # [b, d_model]

    # 重塑为多头形式 [b, h, 1, d_k]
    q = q.view(batch_size, 1, num_heads, d_k).transpose(1, 2)
    k = k.view(batch_size, 1, num_heads, d_k).transpose(1, 2)
    v = v.view(batch_size, 1, num_heads, d_k).transpose(1, 2)

    # 拼接历史KV缓存
    new_K = torch.cat([prev_K, k], dim=2)  # [b, h, m+1, d_k]
    new_V = torch.cat([prev_V, v], dim=2)  # [b, h, m+1, d_k]

    # 计算注意力分数
    scores = torch.matmul(q, new_K.transpose(-2, -1)) / torch.sqrt(torch.tensor(d_k, dtype=torch.float32))
    attn_weights = F.softmax(scores, dim=-1)  # [b, h, 1, m+1]

    # 加权求和
    attn_output = torch.matmul(attn_weights, new_V)  # [b, h, 1, d_k]

    # 重塑回原始维度 [b, h, 1, d_k] → [b, 1, d_model]
    attn_output = attn_output.transpose(1, 2).contiguous().view(batch_size, 1, d_model)
    y = W_o(attn_output).squeeze(1)  # [b, d_model]

    return y, new_K, new_V

### 增量注意力性能分析

b：batch 大小；

n：总共生成n个 token，增量函数循环调用n次；

d：模型隐层维度；h：头数；k=d/h

总算术运算总量：$\Theta(bnd^2)$

总内存访问开销：$\Theta(bn^2d+nd^2)$

第一项$bn^2d$：由每一步都要全量读取KV 缓存带来；

第二项$nd^2$：由 4 个投影参数$W_q,W_k,W_v,W_o$重复读取带来；

访存 / 计算比：$\Theta\left(\dfrac{n}{d}+\dfrac1b\right)$

当生成场景满足 $n\approx d$（上下文很长）或 $b\approx1$（日常单条文本生成）时，比值趋近于 1，显存带宽成为推理瓶颈（GPU 算力闲置、一直在等数据）。

优化$\boldsymbol{\dfrac1b}$很简单：在显存允许前提下，增大 batch 批量；

优化$\boldsymbol {\dfrac{n}{d}}$难度更高：该项来自每一步推理需要完整加载全部历史 KV 张量（KV 体积随序列变长线性膨胀）。传统解法两种：① 限制最大上下文长度n；② 局部注意力 / 历史 KV 压缩，让每个 token 只关注少量历史位置。

本文提出全新正交优化思路：查询Q维持多头维度，把 K、V 的多头 (h) 维度直接抹除，从张量尺寸上压缩 KV 缓存。

原版$\mathrm{K/V}:[b,h,n,k]$，带h个头；

新方案：$\boldsymbol{Q}$保留多头h，$\boldsymbol{K/V}$去掉头维度，$[b,n,d]$，直接砍掉 KV 的h倍体积，大幅降低$bn^2d$项，从源头压小$\frac nd$，和传统局部注意力互不冲突（正交方案）。


## 多查询注意力

多头注意力（MHA）：并行的 h 个注意力头，每个头都有独立的 Q、K、V、输出线性层；

多查询注意力（MQA）：和多头注意力几乎完全一样，唯一区别：所有注意力头共享同一组 K 和 V。

**批量版本：** K = tf.einsum("bmd,dk->bmk", M, P_k) — W_k 形状从 [h,d,k] 降为 [d,k]，K 形状从 [b,h,m,k] 降为 [b,m,k]

**增量版本：** prev_K 形状从 [b,h,m,k] 降为 [b,m,k]，prev_V 从 [b,h,m,v] 降为 [b,m,v]

In [ ]:
import torch
import torch.nn.functional as F

def MultiqueryAttentionBatched(X, M, mask, W_q, W_k, W_v, W_o, num_heads):
    """
    Multi-Query Attention (MQA) - 批量版本
    Q 多头 / K,V 共享无头维度
    params：
        X: [batch_size, seq_len_q, d_model]
        M: [batch_size, seq_len_k, d_model]
        mask: [batch_size, num_heads, seq_len_q, seq_len_k]
        num_heads: 注意力头数
    output：
        Y: [batch_size, seq_len_q, d_model]
    """
    batch_size = X.shape[0]
    d_model = X.shape[-1]
    d_k = d_model // num_heads

    # Q保留多头，K/V共享无头维度
    q = W_q(X)  # [b, n_q, d_model]
    q = q.view(batch_size, -1, num_heads, d_k).transpose(1, 2)  # [b, h, n_q, d_k]

    # K/V 无头维度 - 这是MQA的核心
    k = W_k(M)  # [b, n_k, d_k]
    v = W_v(M)  # [b, n_k, d_k]

    k = k.unsqueeze(1)  # [b, 1, n_k, d_k] → 广播匹配所有头
    
    scores = torch.matmul(q, k.transpose(-2, -1)) / torch.sqrt(torch.tensor(d_k, dtype=torch.float32))
    scores = scores + mask  # 加入掩码
    attn_weights = F.softmax(scores, dim=-1)  # [b, h, n_q, n_k]

    v = v.unsqueeze(1)  # [b, 1, n_k, d_k]
    attn_output = torch.matmul(attn_weights, v)  # [b, h, n_q, d_k]

    attn_output = attn_output.transpose(1, 2).contiguous()  # [b, n_q, h, d_k]
    attn_output = attn_output.view(batch_size, -1, d_model)  # [b, n_q, d_model]
    y = W_o(attn_output)

    return y

def MultiquerySelfAttentionIncremental(x, prev_K, prev_V, W_q, W_k, W_v, W_o, num_heads):
    """
    增量式多查询注意力（MQA）
    params：
        x: [batch_size, d_model]
        prev_K: [batch_size, m, d_k]
        prev_V: [batch_size, m, d_k]
        W_q/W_k/W_v/W_o
        num_heads
    output：
        y: [batch_size, d_model]
        new_K: [batch_size, m+1, d_k]
        new_V: [batch_size, m+1, d_k]
    """
    batch_size, d_model = x.shape
    d_k = d_model // num_heads

    # Q保持多头
    q = W_q(x)  # [b, d_model]
    q = q.view(batch_size, 1, num_heads, d_k).transpose(1, 2)  # [b, h, 1, d_k]

    # K/V 无头维度
    k = W_k(x)  # [b, d_k]
    v = W_v(x)  # [b, d_k]
    k = k.unsqueeze(1)  # [b, 1, d_k]
    v = v.unsqueeze(1)  # [b, 1, d_k]

    # 拼接历史缓存（注意：缓存大小比MHA小h倍）
    new_K = torch.cat([prev_K, k], dim=1)  # [b, m+1, d_k]
    new_V = torch.cat([prev_V, v], dim=1)  # [b, m+1, d_k]

    # 广播K以匹配多头Q
    k_expand = new_K.unsqueeze(1)  # [b, 1, m+1, d_k] → 广播匹配h个头
    scores = torch.matmul(q, k_expand.transpose(-2, -1)) / torch.sqrt(torch.tensor(d_k, dtype=torch.float32))
    attn_weights = F.softmax(scores, dim=-1)  # [b, h, 1, m+1]

    # 广播V以匹配多头
    v_expand = new_V.unsqueeze(1)  # [b, 1, m+1, d_k]
    attn_output = torch.matmul(attn_weights, v_expand)  # [b, h, 1, d_k]

    # 重塑回原始维度
    attn_output = attn_output.transpose(1, 2).contiguous().view(batch_size, 1, d_model)
    y = W_o(attn_output).squeeze(1)  # [b, d_model]

    return y, new_K, new_V

MHA 为什么 KV 体积大？—— 同一个 token 重复生成 h 份 K/V MHA：

$W_k\in[h,d,k]$：h 套完全独立的 K 投影权重

同一个输入 token 向量，经过 h 个不同的 $W_k$，算出 h 份独立 K 特征

K 张量：$\boldsymbol{[b,h,n,k]}$

MQA 的 K = 直接学习完整的单头特征 可学习

MQA让网络直接学习一个完整 K/V：包含了所有需要的语义信息；

### MQA性能分析

MQA 的总算术运算量 和标准多头注意力（MHA），都是 $\Theta(bnd^2)$；

连续生成 n 个词，MQA 的总显存访问量 变成了 $\Theta(bnd + bn^2k + nd^2)$

访存计算比：$\Theta\left(\frac{1}{d} + \frac{n}{dh} + \frac{1}{b}\right)$。最慢速度的 $\frac{n}{d}$ 项，缩小了 h 倍


In [ ]:
import torch
import torch.nn as nn
import time
import numpy as np


batch_size = 4          
d_model = 512           
num_heads = 8           
d_k = d_model // num_heads  
seq_len = 128           
num_warmup = 5          

device = torch.device('cpu')

# MHA Q/K/V都是多头，权重矩阵输出维度为 d_model
W_q_mha = nn.Linear(d_model, d_model, bias=False).to(device)
W_k_mha = nn.Linear(d_model, d_model, bias=False).to(device)
W_v_mha = nn.Linear(d_model, d_model, bias=False).to(device)
W_o_mha = nn.Linear(d_model, d_model, bias=False).to(device)

# MQA Q是多头(输出d_model)，K/V共享无头维度(输出d_k)
W_q_mqa = nn.Linear(d_model, d_model, bias=False).to(device)
W_k_mqa = nn.Linear(d_model, d_k, bias=False).to(device) 
W_v_mqa = nn.Linear(d_model, d_k, bias=False).to(device)  
W_o_mqa = nn.Linear(d_model, d_model, bias=False).to(device)

print("\n权重矩阵初始化完成:")
print(f"  MHA - W_k shape: {W_k_mha.weight.shape}, W_v shape: {W_v_mha.weight.shape}")
print(f"  MQA - W_k shape: {W_k_mqa.weight.shape}, W_v shape: {W_v_mqa.weight.shape}")
print(f"  MQA的K/V权重矩阵缩小了 {num_heads}x")

print("\n")
print("测试 MHA (Multi-Head Attention) 增量解码")

# 初始化KV缓存 - MHA需要为每个头存储独立的K/V
prev_K_mha = torch.zeros(batch_size, num_heads, 0, d_k).to(device)
prev_V_mha = torch.zeros(batch_size, num_heads, 0, d_k).to(device)

# 预热
for _ in range(num_warmup):
    x = torch.randn(batch_size, d_model).to(device)
    y, prev_K_mha, prev_V_mha = MultiheadSelfAttentionIncremental(
        x, prev_K_mha, prev_V_mha, W_q_mha, W_k_mha, W_v_mha, W_o_mha, num_heads
    )

# 重置缓存
prev_K_mha = torch.zeros(batch_size, num_heads, 0, d_k).to(device)
prev_V_mha = torch.zeros(batch_size, num_heads, 0, d_k).to(device)

# 计时测试
mha_times = []
if device.type == 'cuda':
    torch.cuda.synchronize()

start_time = time.time()
for step in range(seq_len):
    x = torch.randn(batch_size, d_model).to(device)
    
    step_start = time.time()
    y, prev_K_mha, prev_V_mha = MultiheadSelfAttentionIncremental(
        x, prev_K_mha, prev_V_mha, W_q_mha, W_k_mha, W_v_mha, W_o_mha, num_heads
    )
    if device.type == 'cuda':
        torch.cuda.synchronize()
    step_time = time.time() - step_start
    mha_times.append(step_time)

mha_total_time = time.time() - start_time

# 计算KV缓存大小（字节）
mha_cache_size = prev_K_mha.numel() * prev_K_mha.element_size() + prev_V_mha.numel() * prev_V_mha.element_size()
mha_cache_size_mb = mha_cache_size / (1024 ** 2)

print(f"MHA 完成!")
print(f"  - 总时间: {mha_total_time*1000:.2f} ms")
print(f"  - 平均每步: {np.mean(mha_times)*1000:.4f} ms")
print(f"  - KV缓存形状: K={prev_K_mha.shape}, V={prev_V_mha.shape}")
print(f"  - KV缓存大小: {mha_cache_size_mb:.2f} MB")



print("测试 MQA (Multi-Query Attention) 增量解码...")


# 初始化KV缓存 - MQA只需要存储单一的K/V（无头维度）
prev_K_mqa = torch.zeros(batch_size, 0, d_k).to(device)
prev_V_mqa = torch.zeros(batch_size, 0, d_k).to(device)

# 预热
for _ in range(num_warmup):
    x = torch.randn(batch_size, d_model).to(device)
    y, prev_K_mqa, prev_V_mqa = MultiquerySelfAttentionIncremental(
        x, prev_K_mqa, prev_V_mqa, W_q_mqa, W_k_mqa, W_v_mqa, W_o_mqa, num_heads
    )

# 重置缓存
prev_K_mqa = torch.zeros(batch_size, 0, d_k).to(device)
prev_V_mqa = torch.zeros(batch_size, 0, d_k).to(device)

# 计时测试
mqa_times = []
if device.type == 'cuda':
    torch.cuda.synchronize()

start_time = time.time()
for step in range(seq_len):
    x = torch.randn(batch_size, d_model).to(device)
    
    step_start = time.time()
    y, prev_K_mqa, prev_V_mqa = MultiquerySelfAttentionIncremental(
        x, prev_K_mqa, prev_V_mqa, W_q_mqa, W_k_mqa, W_v_mqa, W_o_mqa, num_heads
    )
    if device.type == 'cuda':
        torch.cuda.synchronize()
    step_time = time.time() - step_start
    mqa_times.append(step_time)

mqa_total_time = time.time() - start_time

# 计算KV缓存大小（字节）
mqa_cache_size = prev_K_mqa.numel() * prev_K_mqa.element_size() + prev_V_mqa.numel() * prev_V_mqa.element_size()
mqa_cache_size_mb = mqa_cache_size / (1024 ** 2)

print(f"MQA 完成!")
print(f"  - 总时间: {mqa_total_time*1000:.2f} ms")
print(f"  - 平均每步: {np.mean(mqa_times)*1000:.4f} ms")
print(f"  - KV缓存形状: K={prev_K_mqa.shape}, V={prev_V_mqa.shape}")
print(f"  - KV缓存大小: {mqa_cache_size_mb:.2f} MB")


print("对比结果")


speedup = mha_total_time / mqa_total_time
cache_reduction = mha_cache_size / mqa_cache_size

print(f"速度提升: MQA比MHA快 {speedup:.2f}x")
print(f"缓存减少: MQA的KV缓存是MHA的 {1/cache_reduction:.2f}x (理论值: {1/num_heads:.2f}x)")
print(f"内存节省: {(1 - 1/cache_reduction)*100:.1f}%")



权重矩阵初始化完成:
  MHA - W_k shape: torch.Size([512, 512]), W_v shape: torch.Size([512, 512])
  MQA - W_k shape: torch.Size([64, 512]), W_v shape: torch.Size([64, 512])
  MQA的K/V权重矩阵缩小了 8x


测试 MHA (Multi-Head Attention) 增量解码
MHA 完成!
  - 总时间: 78.94 ms
  - 平均每步: 0.5763 ms
  - KV缓存形状: K=torch.Size([4, 8, 128, 64]), V=torch.Size([4, 8, 128, 64])
  - KV缓存大小: 2.00 MB
测试 MQA (Multi-Query Attention) 增量解码...
MQA 完成!
  - 总时间: 63.52 ms
  - 平均每步: 0.4650 ms
  - KV缓存形状: K=torch.Size([4, 128, 64]), V=torch.Size([4, 128, 64])
  - KV缓存大小: 0.25 MB
对比结果
速度提升: MQA比MHA快 1.24x
缓存减少: MQA的KV缓存是MHA的 0.12x (理论值: 0.12x)
内存节省: 87.5%


# GQA: Training Generalized Multi-Query Transformer Models from Multi-Head Checkpoints

## 主要工作
（1）提出了一种将现有多头语言模型检查点升训为MQA模型的方案，仅需原始预训练计算量的5%；

（2）引入分组查询注意力（GQA）——它是多查询注意力的泛化，使用中等数量（多于1个、少于查询头总数）的键值头。实验表明，升训后的GQA在质量上接近多头注意力，同时速度与MQA相当。

### 续训练-uptraining

Checkpoint 权重转换（不用反向传播、不用训练）原生 MHA：$W_k、W_v$ 是多头参数 $ \boldsymbol{[h,d,k]}$，一共h套独立 KV 权重。

MQA 只需要单个$W_k[d,k]、W_v[d,k]$，论文方案：
把全部h个 head 的 K 投影矩阵做均值池化（算术平均），压缩合并成唯一一套 K 权重；V 同理。
<p align="center">
    <img src="./static/GQA_uptraining.png">
</p>

### Grouped-query attention
<p align="center">
    <img src="./static/GQA.png">
</p>

GQA-G表示具有G个组的分组查询注意力。GQA-1（单个组，因此只有单个键值头）等价于MQA；而GQA-H（组数等于头数）等价于MHA。
